# Optimize enhancer specificity with Malinois

[Notebook](enhancer.ipynb) · [Installation](../docs/installation.md)

Design 200-nt enhancers for K562. The objective subtracts the higher of the two off-target predictions from the K562 prediction.

## 1. Load the model and define the objective



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import os
from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = load_model("malinois", device=device)
# The Malinois network takes a 600-nt construct with the enhancer in the middle.
reference = predictor.encoder(["A" * 200]).transpose(1, 2)
left, right = reference[:, :200], reference[:, -200:]


class EnhancerScorer:
    task_names = predictor.task_names
    device = predictor.device

    def score(self, sequences):
        return predictor.score(sequences)

    def score_soft(self, probabilities):
        n = len(probabilities)
        construct = torch.cat([
            left.to(probabilities).expand(n, -1, -1),
            probabilities,
            right.to(probabilities).expand(n, -1, -1),
        ], dim=1)
        return predictor.model(construct.transpose(1, 2))


model = EnhancerScorer()


def specificity(y):
    return y["K562_log2FC"] - torch.maximum(y["HepG2_log2FC"], y["SKNSH_log2FC"])


run = Path("outputs/malinois_k562")
os.makedirs(run, exist_ok=True)




The scoring adapter adds the same fixed flanks to soft inputs as the pretrained predictor uses for sequence strings.

## 2. Generate candidates



In [2]:
seeds = random_seed_sequences(200, 100, seed=42)
designs = generate_designs(
    model, length=200, initial=seeds, objective=specificity,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,26.810638,28.151599
basesearch,100,21.951774,24.559639
beam_search,100,27.149120,30.651983
cross_entropy_method,100,29.337179,30.064563
den,100,25.795024,27.927972
fast_seqprop,100,26.018300,27.823783
genetic_algorithm,100,35.222167,36.291049
ism_greedy,100,28.238555,32.036017
ledidi,100,26.187701,28.949313




## 3. Discover motifs

Discover motifs from nucleotide contributions to the specificity objective.



In [3]:
motifs = call_motifs(designs, model, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-CCTTATCWGY,2705,1065,5549
1,2-AMCAGCTGKT,686,622,1891
2,3-KATGASTCAK,311,284,1139
4,6-TCCARGAAW,68,60,519
5,7-CCTGCAGGCC,70,70,119
6,10-GGAACAGAC,169,163,105
7,12-AGSCTATCAG,521,453,123
12,19-CACACCCAY,14,14,427
16,28-AGTCACCC,38,37,118




## 4. Select a panel

Use the 20th percentile of the unique candidates’ scores as the minimum score for the panel.



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
panel = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
panel[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,GTTACCCCAATGTCATTATCGCTTGCGACACATCTGCAAGAAACTG...,36.291049,1-CCTTATCWGY;2-AMCAGCTGKT;3-KATGASTCAK;6-TCCAR...
658,2,GCTGTCCTGTGCGATAGCGACTAAAAAAAGCGGCGGAAAAACCTTA...,25.924787,1-CCTTATCWGY
607,3,GTTAGTTCCTTGAATTCTCCTTATAGGCTTCCCGCAGAGTCTTATC...,26.132954,1-CCTTATCWGY;2-AMCAGCTGKT;12-AGSCTATCAG;19-CAC...
413,4,GTTGGTCTGTTTTCCAACGATAAGCCATGACTCATACAAGCTGGCG...,27.059088,1-CCTTATCWGY;3-KATGASTCAK;7-CCTGCAGGCC
229,5,TAGTCCTTATCGCGCCTGCATCTGGTGACCGTTGATTAGTCACCCG...,29.070642,1-CCTTATCWGY;2-AMCAGCTGKT;3-KATGASTCAK;28-AGTC...
854,6,CCTTATCTTTCAAGTAGCAGACGTTATTCACAACCGTTATCACCCA...,24.165561,1-CCTTATCWGY;2-AMCAGCTGKT;6-TCCARGAAW;7-CCTGCA...
624,7,GCTGGCCCTGATAAGGCTGGTGACATTTCCCCTAGGGCGTCTGTGC...,26.026566,1-CCTTATCWGY;2-AMCAGCTGKT
343,8,TCTTGCGCGAGGGTGCCTTATCTGTTCCTTGCAGCACTGTGGGGTT...,27.608120,1-CCTTATCWGY;6-TCCARGAAW
443,9,TATGGTGCGGTTAGATAAGTTCCTTGCAGATGTGAATGACTCGGCA...,26.871327,1-CCTTATCWGY;3-KATGASTCAK;6-TCCARGAAW;19-CACAC...
794,10,GTTATCATGACTCCCTATCTAAGTTATGACGCAACAGGGTGTGCTG...,25.250774,1-CCTTATCWGY;3-KATGASTCAK
